## Step 1: Setup and Data Preparation

In [31]:
%pip install -U -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-openai \
    langchain-pinecone \
    pinecone \
    pypdf \
    python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [32]:
# Import necessary libraries
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys are loaded
for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


### 1a. Load both sources (PDF and Podcast)

In [33]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / r"C:\Users\antun\ai-project\W4\Day-2\lab-relevance-scoring-rerankers\podcast.txt"
PDF_PATH = DATA_DIR / r"C:\Users\antun\ai-project\W4\Day-2\lab-relevance-scoring-rerankers\eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'C:\\Users\\antun\\ai-project\\W4\\Day-2\\lab-relevance-scoring-rerankers\\eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


### 1b. Chunk with metadata

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents into chunks and give every chunk a clean, consistent metadata schema."""
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,        # "podcast" or "regulation" -> used for filtering in Step 5
            "source_name": source_name,        # human-readable name -> used for citations
            "chunk_id": f"{source_type}-{i}",  # unique id -> lets a reviewer trace an exact chunk
        }
        # PDF pages are 0-indexed in the loader; store the page number a human would see
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1
        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


### 1c. Inspect your chunks

In [35]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

{'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'page': 21}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of no ...
--------------------------------------------------------------------------------


### 1d. Embeddings model

In [36]:
from langchain_openai import OpenAIEmbeddings

# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed the text from the first chunk
vector = embeddings.embed_query(all_chunks[0].page_content)

# Print the vector length
print("Vector length:", len(vector))

Vector length: 1536


## Step 2: Embeddings + Baseline Retrieval

In [37]:
# Create (or connect to) a Pinocone index. Use dimension 1536 and metric cosine
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# Initialize Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create or connect to an index
index_name = "rag-intro-index"

# Check if index exists, create if not
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,  # OpenAI text-embedding-3-small dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Created index: {index_name}")
else:
    print(f"Index {index_name} already exists")


Index rag-intro-index already exists


In [38]:
# Create vector store using LangChain / embeds every chunk step
vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)

print("Documents added to Pinecone vector store")

Documents added to Pinecone vector store


In [39]:
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [40]:
def baseline_search(query: str, k: int = 5):
    """Performs a standard similarity search and returns a list of (Document, score) pairs."""
    return vectorstore.similarity_search_with_score(query, k=k)


baseline_results = {}

for q in TEST_QUERIES:
    results = baseline_search(q, k=5)
    baseline_results[q] = results
    show_results(q, results, label="Baseline Search")




=== Baseline Search | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.699  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#4  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#5  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'

=== Baseline Search | Query: What are the seven requirements for trustworthy AI?
#1  score

## Step 3: LLM Relevance Scoring [OPTIONAL (Advanced)]


In [41]:
import re
from langchain_openai import ChatOpenAI

# Initialize gpt-4o-mini with temperature=0 for consistent scoring
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def llm_relevance_score(query: str, doc_content: str) -> float:
    """Prompts gpt-4o-mini to rate passage relevance on a scale of 0.0 to 1.0."""
    prompt = f"""You are an evaluator assessing the relevance of a document passage to a search query.

Query: {query}

Passage:
{doc_content}

Instructions:
Rate how relevant this passage is to answering the query on a continuous scale from 0.0 to 1.0.
- 1.0 = Directly and accurately answers the query.
- 0.5 = Mentions relevant concepts or context, but does not fully answer.
- 0.0 = Completely irrelevant.

Output ONLY a single floating-point number between 0.0 and 1.0. Do not include any other text."""

    try:
        response = llm.invoke(prompt).content.strip()
        match = re.search(r"\b(0(?:\.\d+)?|1(?:\.0+)?)\b", response)
        if match:
            return float(match.group(1))
    except Exception:
        pass

    return 0.0


def llm_rerank(query, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7):
    """Retrieve k_initial candidates, score each chunk with an LLM, combine scores,

    and return the top k_final as (doc, combined_score) pairs.
    """
    # 1. Retrieve initial candidate pool
    candidates = baseline_search(query, k=k_initial)

    # 2. Score each chunk and compute combined score
    reranked = []
    for doc, sim_score in candidates:
        llm_score = llm_relevance_score(query, doc)
        combined_score = (w_sim * sim_score) + (w_llm * llm_score)
        reranked.append((doc, combined_score))

    # 3. Sort descending by combined score and return top k_final
    reranked.sort(key=lambda item: item[1], reverse=True)
    return reranked[:k_final]

In [42]:
for q in TEST_QUERIES:
    # Baseline top 5
    baseline = baseline_search(q, k=5)
    show_results(q, baseline, label="Baseline Top 5")

    # LLM reranked top 5 (retrieving 10 first)
    reranked = llm_rerank(q, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7)
    show_results(q, reranked, label="LLM Reranked Top 5")


=== Baseline Top 5 | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.699  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#4  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#5  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'

=== LLM Reranked Top 5 | Query: What AI practices are prohibited under the EU AI Act?
#1  s

## Step 4: Dedicated Reranker (Cross-Encoder or Cohere) [OPTIONAL (Advanced)]

In [43]:
%pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

# Initialize the cross-encoder model (runs once)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def crossencoder_rerank(query, k_initial=10, k_final=5):
    # 1. Retrieve k_initial candidates using baseline_search when available
    if "baseline_search" in globals() and callable(baseline_search):
        candidates = baseline_search(query, k=k_initial)
    elif "vectorstore" in globals():
        candidates = [
            (doc, score)
            for doc, score in vectorstore.similarity_search_with_score(query, k=k_initial)
        ]
    else:
        raise NameError("No retrieval index is available. Run the vectorstore setup cells first.")

    if not candidates:
        return []

    # 2. Score (query, chunk) pairs
    pairs = [(query, doc.page_content) for doc, _ in candidates]
    scores = reranker.predict(pairs)

    # 3. Combine docs with their cross-encoder score and sort descending
    scored_docs = [
        (doc, float(score)) for (doc, _), score in zip(candidates, scores)
    ]
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    # 4. Return top k_final as (doc, score) pairs
    return scored_docs[:k_final]


Note: you may need to restart the kernel to use updated packages.


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2126.50it/s]


In [44]:
for q in TEST_QUERIES:
    # Baseline
    baseline = baseline_search(q, k=5)
    show_results(q, baseline, label="Baseline Top 5")

    # Step 3: LLM Rerank
    llm_results = llm_rerank(q, k_initial=10, k_final=5)
    show_results(q, llm_results, label="Step 3: LLM Rerank Top 5")

    # Step 4: Cross-Encoder Rerank
    ce_results = crossencoder_rerank(q, k_initial=10, k_final=5)
    show_results(q, ce_results, label="Step 4: Cross-Encoder Top 5")


=== Baseline Top 5 | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.699  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#4  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#5  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'

=== Step 3: LLM Rerank Top 5 | Query: What AI practices are prohibited under the EU AI Act?

## Step 5: Metadata Filtering

In [45]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. Initialize embeddings (use the same model you used when uploading)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Set your existing Pinecone index name
index_name = "rag-intro-index"

# 3. Connect to the existing index
vectorstore = PineconeVectorStore.from_existing_index(
    index_name=index_name, embedding=embeddings
)

In [46]:
# First part: Filter by Source (pdf vs podcast)
test_query = "What AI practices are prohibited under the EU AI Act?"

# Pdf
regulation_results = vectorstore.similarity_search_with_score(
    test_query, k=3, filter={"source_type": {"$eq": "regulation"}}
)
show_results(test_query, regulation_results, label="Source: Regulation Only")

# Podcast 
podcast_results = vectorstore.similarity_search_with_score(
    test_query, k=3, filter={"source_type": {"$eq": "podcast"}}
)
show_results(test_query, podcast_results, label="Source: Podcast Only")


=== Source: Regulation Only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.699  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'

=== Source: Podcast Only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.519  [podcast | podcast-4]  "a pixel, it doesn't recognize or can be easily hacked, then it causes harm. An intentional harm is still harm. So lawful"
#2  score=0.519  [podcast | podcast-4]  "a pixel, it doesn't recognize or can be easily hacked, then it causes harm. An intentional harm is still harm. So lawful

In [47]:
# Part 2: Filtering on Missing Field

attempt_results = vectorstore.similarity_search_with_score(
    test_query, k=3, filter={"article": {"$eq": "Article 5"}}
)
print(f"Results returned: {len(attempt_results)}")


Results returned: 0


In [48]:
# Part 3: Updating Metadata Ingestion & Re-uploading
import re


def enrich_chunk_metadata(chunk, page_num=None, source_type="regulation"):
    """Enriches chunk metadata with section and article tags without storing None values."""
    text = chunk.page_content
    meta = {"source_type": source_type}

    if page_num is not None:
        meta["page"] = page_num

    if source_type == "regulation":
        # 1. Classify section based on page ranges and structural headers
        if page_num is not None and page_num <= 44:
            meta["section"] = "recital"
        elif "ANNEX" in text[:150].upper():
            meta["section"] = "annex"
        else:
            meta["section"] = "article"

        # 2. Detect specific Article headings using regex (e.g. "Article 5")
        article_match = re.search(r"(?:^|\n)\s*Article\s+(\d+)", text)
        if article_match:
            meta["article"] = f"Article {article_match.group(1)}"

    # Crucial: Pinecone rejects None values. Only assign non-None fields.
    chunk.metadata = {k: v for k, v in meta.items() if v is not None}
    return chunk

In [49]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

# Initialize Pinecone and Embeddings
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index_name = "eu-ai-act-rag"  # your existing index name
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 1. Delete old index
if index_name in [idx.name for idx in pc.list_indexes()]:
    print(f"Deleting existing index: {index_name}...")
    pc.delete_index(index_name)
    time.sleep(5)  # Wait for cloud cleanup

# 2. Recreate index
print(f"Creating new index: {index_name}...")
pc.create_index(
    name=index_name,
    dimension=1536,  # 1536 for text-embedding-3-small
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

# 3. Enrich your split chunks using the new function
# (Assumes all_chunks is your list of LangChain Document objects from Step 1)
enriched_chunks = []
for doc in all_chunks:
    page = doc.metadata.get("page")
    src = doc.metadata.get("source_type", "regulation")
    enriched_chunks.append(enrich_chunk_metadata(doc, page_num=page, source_type=src))

# 4. Upload to Pinecone
vectorstore = PineconeVectorStore.from_documents(
    documents=enriched_chunks, embedding=embeddings, index_name=index_name
)
print("Re-indexing complete!")

Deleting existing index: eu-ai-act-rag...
Creating new index: eu-ai-act-rag...
Re-indexing complete!


In [50]:
# Part 4: test the New Granular Filter

# 1. Restrict search exclusively to operative Articles (ignoring recitals & annexes)
article_results = vectorstore.similarity_search_with_score(
    "What AI practices are prohibited?",
    k=3,
    filter={"section": {"$eq": "article"}},
)
show_results(
    "What AI practices are prohibited?",
    article_results,
    label="Filter: Operative Articles Only",
)

# 2. Pinpoint Article 5 directly
art5_results = vectorstore.similarity_search_with_score(
    "prohibited practices", k=2, filter={"article": {"$eq": "Article 5"}}
)
show_results(
    "prohibited practices", art5_results, label="Filter: Article 5 Only"
)


=== Filter: Operative Articles Only | Query: What AI practices are prohibited?
#1  score=0.739  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.568  [regulation | p.51.0]  'materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an '
#3  score=0.563  [regulation | p.113.0]  'Article 18 of Regulation (EU) 2019/1020 shall apply mutatis mutandis to the providers of the general-purpose AI model, \n'

=== Filter: Article 5 Only | Query: prohibited practices


## Step 6: Complete RAG Pipeline with Reranking

In [51]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Format Context with Inline Source Tags
def format_context(results) -> str:
    """Formats (doc, score) pairs or Document lists into labeled context blocks."""
    if not results:
        return "No relevant context found."

    formatted_chunks = []
    for item in results:
        doc = item[0] if isinstance(item, (tuple, list)) else item
        m = getattr(doc, "metadata", {}) or {}

        source = m.get("source_type", "EU AI Act")
        location = f"p.{m['page']}" if "page" in m else m.get("chunk_id", "n/a")
        tag = f"[{source}, {location}]"

        formatted_chunks.append(f"{tag}
{doc.page_content.strip()}")

    return "

".join(formatted_chunks)

# Prompt with Citation Rules & Strict Fallback
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a specialized legal and technical assistant answering questions based strictly on the provided context.

Rules:
1. Answer the question using ONLY the context provided below.
2. Every factual claim must include an inline citation tag matching the context header (e.g., [regulation, p.31] or [EU AI Act, p.51]).
3. If the context does not contain enough information to answer the question, say exactly:
   "I don't know based on the sources."
   Do NOT extrapolate, hallucinate, or rely on outside knowledge.""",
        ),
        (
            "human",
            """Context:
{context}

Question: {question}

Answer:""",
        ),
    ]
)

# ---------------------------------------------------------------------------
# Modular Retrieval Switch (1-Line Toggle)
# ---------------------------------------------------------------------------
def retrieve(query: str):
    """Return retrieval results in a way that works even if earlier cells were skipped."""
    if "baseline_search" in globals() and callable(baseline_search):
        return baseline_search(query, k=5)
    if "vectorstore" in globals():
        return vectorstore.similarity_search_with_score(query, k=5)
    raise NameError("No retrieval function is available. Run the vectorstore setup cells first.")

# Option A: Baseline vector search
# retrieve = lambda q: baseline_search(q, k=5)

# Option B: Cross-encoder rerank (Step 4)
# retrieve = lambda q: crossencoder_rerank(q, k_initial=10, k_final=5)

# Option C: LLM rerank (Step 3)
# retrieve = lambda q: llm_rerank(q, k_initial=10, k_final=5)
# ---------------------------------------------------------------------------

# Build LCEL Chain
rag_chain = (
    {
        "context": RunnableLambda(retrieve) | format_context,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# Testing the Pipeline
# Test 1: Grounded Answer with Citations
query_1 = "What AI practices are prohibited under the EU AI Act?"
response_1 = rag_chain.invoke(query_1)
print(response_1)

The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective or effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision [regulation, p.51.0].

2. AI-enabled manipulative techniques that persuade persons to engage in unwanted behaviours or deceive them by nudging them into decisions in a way that subverts and impairs their autonomy, decision-making, and free choices, particularly when significant harms are likely to occur, affecting physical, psychological health, or financial interests [regulation, p.9.0]. 

3. Practices that contradict Union values of respect for human dignity, freedom, equality, democracy, and the rule of law, including manipulative, exploitative, and soc

In [53]:
# Test 2: Fallback Guardrai
query_2 = "What does the EU AI Act say about quantum computing cooling systems?"
response_2 = rag_chain.invoke(query_2)
print(response_2)

I don't know based on the sources.


## Step 7: Evaluate: With vs. Without Reranking 

In [54]:
import time


def generate_answer(query: str, search_fn) -> tuple[str, list, float]:
    """Runs retrieval with search_fn, formats context, calls the LLM,

    and returns the generated answer, retrieved docs, and total latency.
    """
    start_time = time.perf_counter()

    # 1. Retrieve candidates
    results = search_fn(query)

    # 2. Format context with source metadata tags
    context = format_context(results)

    # 3. Generate answer via the LCEL prompt & model
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": query})

    elapsed = time.perf_counter() - start_time
    return answer, results, elapsed


# Run evaluation loop across test queries
evaluation_logs = {}

for q in TEST_QUERIES:
    print("\n" + "=" * 80)
    print(f"QUERY: {q}")
    print("=" * 80)

    # 1. Baseline retrieval (k=5)
    base_ans, base_docs, base_time = generate_answer(
        q, lambda query: baseline_search(query, k=5)
    )

    # 2. Cross-encoder reranked retrieval (k_initial=10 -> k_final=5)
    rerank_ans, rerank_docs, rerank_time = generate_answer(
        q, lambda query: crossencoder_rerank(query, k_initial=10, k_final=5)
    )

    # Print comparative report
    print(f"\n--- [Baseline Search] ({base_time:.2f}s) ---")
    if base_docs:
        top_meta = getattr(base_docs[0][0], "metadata", {})
        print(
            f"Top-1 Chunk: [{top_meta.get('source_type')} | p.{top_meta.get('page') or top_meta.get('chunk_id')}]"
        )
    print(f"Answer:\n{base_ans}\n")

    print(f"--- [Cross-Encoder Reranked] ({rerank_time:.2f}s) ---")
    if rerank_docs:
        top_meta = getattr(rerank_docs[0][0], "metadata", {})
        print(
            f"Top-1 Chunk: [{top_meta.get('source_type')} | p.{top_meta.get('page') or top_meta.get('chunk_id')}]"
        )
    print(f"Answer:\n{rerank_ans}\n")

    evaluation_logs[q] = {
        "baseline": {"time": base_time, "answer": base_ans, "docs": base_docs},
        "reranked": {
            "time": rerank_time,
            "answer": rerank_ans,
            "docs": rerank_docs,
        },
    }


QUERY: What AI practices are prohibited under the EU AI Act?

--- [Baseline Search] (3.66s) ---
Top-1 Chunk: [regulation | p.51.0]
Answer:
The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective or effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an informed decision [regulation, p.51.0]. 

Additionally, practices that lead to economic or financial harms to consumers through unfair commercial practices are prohibited under all circumstances, irrespective of whether they are implemented through AI systems or otherwise [regulation, p.9.0].

--- [Cross-Encoder Reranked] (3.64s) ---
Top-1 Chunk: [regulation | p.12.0]
Answer:
The following AI practices are prohibited under the EU AI Act: